In [0]:
%run "../00_Configuration/00_parametres"

In [0]:
from pyspark.sql.functions import col, round, when, current_timestamp, lit

print("🏆 Génération du score de potentiel (Couche GOLD)...\n")

try:
    # 1. Chargement des tables Silver propres
    df_stations = spark.table(f"{DB_SILVER}.slv_stations")
    df_communes = spark.table(f"{DB_SILVER}.slv_communes")
    df_trafic = spark.table(f"{DB_SILVER}.slv_trafic")
    
    # 2. Logique de Scoring (Exemple métier pour Afriquia)
    # Nous allons attribuer une note sur 100 basée sur la densité et le trafic
    # Poids : 40% Densité Population | 60% Intensité Trafic
    
    # Note : Dans un projet réel, on ferait une jointure spatiale. 
    # Pour ton PFE, nous allons agréger les indicateurs par zone.
    
    df_gold = df_communes.withColumn(
        "score_demographique", 
        round((col("densite") / 5000) * 40, 2) # Normalisation sur 40 points
    ).withColumn(
        "score_final",
        when(col("score_demographique") > 40, 40).otherwise(col("score_demographique")) + lit(45) # Bonus simulant le trafic routier
    )
    
    # 3. Détermination de la priorité d'investissement
    df_final = df_gold.withColumn(
        "decision_afriquia",
        when(col("score_final") >= 75, "🚀 Priorité Haute - Ouverture Immédiate")
        .when(col("score_final") >= 50, "📈 Priorité Moyenne - Étude de terrain")
        .otherwise("🔍 Observation - Zone saturée ou faible flux")
    ).select(
        "commune", 
        "population", 
        "densite", 
        "score_final", 
        "decision_afriquia"
    ).orderBy(col("score_final").desc())

    # 4. Sauvegarde de la table finale pour ton Dashboard
    df_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DB_GOLD}.gld_potentiel_implantation")
    
    print("✅ Table GOLD créée : gld_potentiel_implantation")
    display(df_final.limit(10))

except Exception as e:
    print(f"❌ Erreur lors du calcul Gold : {e}")